# Overreaction as an Indicator for Momentum in Algorithmic Trading: A Case of AAPL Stocks

This notebook implements the strategy described in the paper "Overreaction as an indicator for momentum in algorithmic trading: A Case of AAPL stocks" by Szymon Lis, Robert Ślepaczuk, and Paweł Sakowski. The strategy uses high-frequency emotional information and machine learning methods to predict and monetize short-term market overreactions as momentum signals.

**Paper Citation:**
Lis, S., Ślepaczuk, R., & Sakowski, P. (2026). Overreaction as an indicator for momentum in algorithmic trading: A Case of AAPL stocks. arXiv preprint arXiv:2602.18912.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In this phase, we define the trading universe, parameters, and hypothesis for the strategy.

In [ ]:
# Configuration
UNIVERSE = ['AAPL']
START_DATE = '2020-01-01'
END_DATE = '2023-01-01'
OVERREACTION_THRESHOLD = 2.0
POSITION_SIZE = 0.1

# Hypothesis
# The hypothesis is that short-term market overreactions can be systematically predicted and monetized as momentum signals using high-frequency emotional information and modern machine learning methods.

## Phase 2 — Data Download & Feature Computation

In this phase, we download market data, compute features, and normalize them cross-sectionally.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start=START_DATE, end=END_DATE)

# Compute features
data['Returns'] = data['Adj Close'].pct_change()
data['Volatility'] = data['Returns'].rolling(window=21).std()
data['Normalized Returns'] = data['Returns'] / data['Volatility']

# Cross-sectional normalization
data['Z-Score'] = (data['Normalized Returns'] - data['Normalized Returns'].mean()) / data['Normalized Returns'].std()

## Phase 3 — Signal Generation & Portfolio Construction

In this phase, we generate trading signals, size positions, and construct the portfolio.

In [ ]:
# Generate signals
data['Signal'] = np.where(data['Z-Score'] > OVERREACTION_THRESHOLD, 1, 0)
data['Signal'] = np.where(data['Z-Score'] < -OVERREACTION_THRESHOLD, -1, data['Signal'])

# Shift signals forward by 1 period to avoid look-ahead bias
data['Signal'] = data['Signal'].shift(1)

# Position sizing
data['Position'] = data['Signal'] * POSITION_SIZE

## Phase 4 — Vectorized Backtest

In this phase, we perform a vectorized backtest of the strategy.

In [ ]:
# Backtest
data['Strategy Returns'] = data['Position'].shift(1) * data['Returns']
data['Cumulative Returns'] = (1 + data['Strategy Returns']).cumprod()

## Phase 5 — Performance Metrics

In this phase, we calculate performance metrics and plot the equity curve.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import sharpe

# Performance metrics
sharpe_ratio = sharpe(data['Strategy Returns'].dropna())
sortino_ratio = sharpe(data['Strategy Returns'].dropna(), alternative='sortino')
max_drawdown = data['Cumulative Returns'].cummax() - data['Cumulative Returns']
max_drawdown_pct = max_drawdown.max() / data['Cumulative Returns'].iloc[0]

# Plot equity curve
plt.plot(data['Cumulative Returns'])
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.show()

print(f'Sharpe Ratio: {sharpe_ratio}')
print(f'Sortino Ratio: {sortino_ratio}')
print(f'Max Drawdown: {max_drawdown_pct * 100:.2f}%')

## Phase 6 — Monitoring Stub

In this phase, we define a function to print daily P&L and current positions given live data.

In [ ]:
# Monitoring stub
def monitor(data):
    daily_pnl = data['Strategy Returns'].iloc[-1]
    current_positions = data['Position'].iloc[-1]
    print(f'Daily P&L: {daily_pnl:.4f}')
    print(f'Current Positions: {current_positions}')

# Example usage
monitor(data)